In [ ]:
import math
import re
import numpy as np
from collections import defaultdict

def parse_codon_usage_database(filepath):
    """
    Parse codon usage database format from a text file into a dictionary.
    Convert RNA codons (U) to DNA codons (T), and extract frequencies.
    """
    codon_usage = {}
    with open(filepath, "r") as file:
        for line in file:
            # Skip empty lines
            line = line.strip()
            if not line:
                continue
            # Split by '(' and ')'
            entries = re.split(r'[()]', line)
            for i in range(0, len(entries), 2):  # Process codons and their frequencies
                entries[i] = entries[i].strip()
                if not entries[i]:
                    break
                codon = entries[i][0:3]
                frequency = float(entries[i][3:])
                codon = codon.replace("U", "T")  # Convert RNA codon to DNA
                codon_usage[codon] = frequency
    return codon_usage

def compute_relative_adaptiveness(codon_usage):
    """
    Compute the relative adaptiveness (w) for each codon.
    """
    relative_adaptiveness = {}
    # Group codons by their amino acids
    amino_acid_to_codons = defaultdict(list)
    for codon, freq in codon_usage.items():
        amino_acid = codon_to_amino_acid(codon)
        amino_acid_to_codons[amino_acid].append((codon, freq))
    
    # Calculate relative adaptiveness for each codon
    for amino_acid, codons in amino_acid_to_codons.items():
        max_freq = max(freq for codon, freq in codons)  # Find the maximum frequency for the amino acid
        if len(codons) > 1:
            for codon, freq in codons:
                relative_adaptiveness[codon] = freq / max_freq
            
    return relative_adaptiveness, amino_acid_to_codons

def codon_to_amino_acid(codon):
    """
    Convert a codon to its corresponding amino acid using the standard genetic code.
    """
    genetic_code = {
        "TTT": "F", "TTC": "F", "TTA": "L", "TTG": "L",
        "CTT": "L", "CTC": "L", "CTA": "L", "CTG": "L",
        "ATT": "I", "ATC": "I", "ATA": "I", "ATG": "M",
        "GTT": "V", "GTC": "V", "GTA": "V", "GTG": "V",
        "TCT": "S", "TCC": "S", "TCA": "S", "TCG": "S",
        "CCT": "P", "CCC": "P", "CCA": "P", "CCG": "P",
        "ACT": "T", "ACC": "T", "ACA": "T", "ACG": "T",
        "GCT": "A", "GCC": "A", "GCA": "A", "GCG": "A",
        "TAT": "Y", "TAC": "Y", "TAA": "*", "TAG": "*",
        "CAT": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
        "AAT": "N", "AAC": "N", "AAA": "K", "AAG": "K",
        "GAT": "D", "GAC": "D", "GAA": "E", "GAG": "E",
        "TGT": "C", "TGC": "C", "TGA": "*", "TGG": "W",
        "CGT": "R", "CGC": "R", "CGA": "R", "CGG": "R",
        "AGT": "S", "AGC": "S", "AGA": "R", "AGG": "R",
        "GGT": "G", "GGC": "G", "GGA": "G", "GGG": "G"
    }
    return genetic_code.get(codon, None)

def compute_cai(sequence, relative_adaptiveness):
    """
    Compute the Codon Adaptation Index (CAI) for a query DNA sequence.
    """
    codons = [sequence[i:i+3] for i in range(0, len(sequence), 3)]
    weights = []
    
    for codon in codons:
        if codon in relative_adaptiveness:
            weights.append(relative_adaptiveness[codon])
        #else:
        #    weights.append(0)  # Unknown codons are assigned a weight of 0
        #    print('can not find')
    
    # Compute the geometric mean of the weights
    mysum = sum(np.log(weights))
    cai = np.e**(mysum/len(weights))
    return cai


def read_fasta_file(filepath):
    """
    Read sequences from a FASTA file.
    Returns a dictionary where keys are sequence headers and values are sequences.
    """
    sequences = {}
    with open(filepath, "r") as file:
        header = None
        sequence = []
        for line in file:
            line = line.strip()
            if line.startswith(">"):  # Header line
                if header:
                    sequences[header] = "".join(sequence)
                header = line[1:]  # Remove '>'
                sequence = []
            else:
                sequence.append(line.upper())  # Sequence lines
        if header:
            sequences[header] = "".join(sequence)  # Add last sequence
    return sequences


'\n# Example usage\nif __name__ == "__main__":\n    # Step 1: Read the codon usage table from a file\n    codon_usage_file = "codon_usage_database.txt"  # Replace with your file path\n    codon_usage = parse_codon_usage_database(codon_usage_file)\n\n    # Step 2: Compute relative adaptiveness\n    relative_adaptiveness = compute_relative_adaptiveness(codon_usage)\n\n    # Step 3: Define a query DNA sequence\n    query_sequence = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG"\n\n    # Step 4: Compute the CAI\n    cai = compute_cai(query_sequence, relative_adaptiveness)\n    print(f"Codon Adaptation Index (CAI): {cai:.4f}")\n'

In [ ]:
# Step 1: Read the codon usage table from a file
codon_usage_file = "ref_codon_usage_table/Homo sapiens.txt"  # Replace with your file path
codon_usage = parse_codon_usage_database(codon_usage_file)

# Step 2: Compute relative adaptiveness
relative_adaptiveness, amino_acid_to_codons = compute_relative_adaptiveness(codon_usage)

# Step 3: Define a query DNA sequence
# After download the data from GISAID, please extract the sequences for different lineages
# based on the lineage information at genomic_features/all_features_seqname_lineage_year.tsv
fasta_file = "lineage_seq/Ca2_genome.fas"  # Replace with your FASTA file
sequences = read_fasta_file(fasta_file)
    
# Step 4: Compute and display CAI for each sequence
print("Codon Adaptation Index (CAI) for each sequence:")
seqname2cai = {}
for header, sequence in sequences.items():
    cai = compute_cai(sequence, relative_adaptiveness)
    #print(f"{header}: {cai:.4f}")
    seqname2cai[header] = cai
    
# Save to a plain text file
with open("../../CAI/cai_Ca2.csv", "w") as file:
    for key, value in seqname2cai.items():
        file.write(f"{key},{value}\n")

Codon Adaptation Index (CAI) for each sequence:
